# Feature Engineering Pipeline — Group 13
**Input:** Raw Chicago crime data (2015–2024 parquet + 2025 CSV)

**Output:** `chicago_features_processed.parquet` — 30 engineered features, ready for model training

**Design principle:** Feature engineering is **separate** from model training.
- Run this notebook **once** → saves processed dataset
- Model training notebook loads the saved parquet
- This avoids recomputing slow rolling windows on every training run

**Key improvements over Phase 2 baseline:**
| | Baseline (Phase 2) | This pipeline |
|---|---|---|
| Lag features | Raw sum | Rolling **mean** (scale-independent) |
| Windows | 7d, 30d | 7d, 30d, **90d** |
| Surge detection | None | **Surge ratio** (7d/30d) |
| Trend | None | **Trend** (7d − 30d) |
| Cross-tier | None | **2 ratio features** |
| Composite risk | None | **4 weighted composites** |
| Total features | 12 | **30** |


## Step 0: Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
FOLDER      = 'G:/My Drive/To be - AI Medical Engineer/Claude_app_Project/IT5006 Group Project Final'
TRAIN_FILE  = os.path.join(FOLDER, 'chicago_crimes.parquet')
TEST_FILE   = os.path.join(FOLDER, 'chicago_crimes_2025_testing.csv')
OUT_FILE    = os.path.join(FOLDER, 'chicago_features_processed.parquet')
SAVE_DIR    = os.path.join(FOLDER, 'Saved_Models')
os.makedirs(SAVE_DIR, exist_ok=True)

TIERS = [
    'Tier 1 - Lethal',
    'Tier 2 - Personal Violence',
    'Tier 3 - Property',
    'Tier 4 - Public Order'
]
TARGET_COLS = ['y_tier1', 'y_tier2', 'y_tier3', 'y_tier4']

SEVERITY_MAPPING = {
    'HOMICIDE':                      'Tier 1 - Lethal',
    'CRIMINAL SEXUAL ASSAULT':        'Tier 1 - Lethal',
    'CRIM SEXUAL ASSAULT':            'Tier 1 - Lethal',
    'KIDNAPPING':                     'Tier 1 - Lethal',
    'HUMAN TRAFFICKING':              'Tier 1 - Lethal',
    'BATTERY':                        'Tier 2 - Personal Violence',
    'ASSAULT':                        'Tier 2 - Personal Violence',
    'ROBBERY':                        'Tier 2 - Personal Violence',
    'SEX OFFENSE':                    'Tier 2 - Personal Violence',
    'OFFENSE INVOLVING CHILDREN':     'Tier 2 - Personal Violence',
    'STALKING':                       'Tier 2 - Personal Violence',
    'THEFT':                          'Tier 3 - Property',
    'MOTOR VEHICLE THEFT':            'Tier 3 - Property',
    'BURGLARY':                       'Tier 3 - Property',
    'ARSON':                          'Tier 3 - Property',
    'DECEPTIVE PRACTICE':             'Tier 3 - Property',
    'CRIMINAL DAMAGE':                'Tier 3 - Property',
    'CRIMINAL TRESPASS':              'Tier 3 - Property',
    'INTIMIDATION':                   'Tier 3 - Property',
    'NARCOTICS':                      'Tier 4 - Public Order',
    'PROSTITUTION':                   'Tier 4 - Public Order',
    'PUBLIC PEACE VIOLATION':         'Tier 4 - Public Order',
    'GAMBLING':                       'Tier 4 - Public Order',
    'LIQUOR LAW VIOLATION':           'Tier 4 - Public Order',
    'INTERFERENCE WITH PUBLIC OFFICER': 'Tier 4 - Public Order',
    'WEAPONS VIOLATION':              'Tier 4 - Public Order',
}

print('Config ready.')
print(f'  Train file : {TRAIN_FILE}')
print(f'  Test file  : {TEST_FILE}')
print(f'  Output     : {OUT_FILE}')

## Step 1: Load & Combine Raw Data
> **Why combine first?**
> 2025 lag features (e.g. January 2025 7-day lag) must look back into December 2024.
> Processing separately would give wrong lag values for the first 90 days of 2025.


In [ ]:
COLS = ['Case Number', 'Date', 'Primary Type', 'Latitude', 'Longitude']

print('Loading training data (2015-2024)...')
df_train_raw = pd.read_parquet(TRAIN_FILE, columns=COLS)
print(f'  Train raw shape: {df_train_raw.shape}')

print('Loading validation data (2025)...')
df_test_raw = pd.read_csv(TEST_FILE, usecols=COLS)
print(f'  Test raw shape : {df_test_raw.shape}')

# Combine
df_all = pd.concat([df_train_raw, df_test_raw], ignore_index=True)

# Parse dates
df_all['Date'] = pd.to_datetime(df_all['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df_all = df_all.dropna(subset=['Date', 'Latitude', 'Longitude'])
df_all = df_all.drop_duplicates(subset=['Case Number'], keep='first')
df_all = df_all[(df_all['Date'].dt.year >= 2015) & (df_all['Date'].dt.year <= 2025)]

# Severity tier mapping
df_all['Severity_Tier'] = df_all['Primary Type'].map(SEVERITY_MAPPING).fillna('Other')

# Grid ID (same spatial unit as Phase 2 baseline)
df_all['grid_id'] = (
    np.floor(df_all['Latitude']  / 0.0045).astype(int).astype(str) + '_' +
    np.floor(df_all['Longitude'] / 0.0060).astype(int).astype(str)
)

# Normalize to daily level
df_all['Date'] = df_all['Date'].dt.normalize()

print(f'\nCombined shape  : {df_all.shape}')
print(f'Date range      : {df_all["Date"].min().date()} -> {df_all["Date"].max().date()}')
print(f'Year counts:\n{df_all["Date"].dt.year.value_counts().sort_index()}')

## Step 2: Build Daily Grid Panel

In [ ]:
print('Building daily grid panel...')

df_daily = (
    df_all[df_all['Severity_Tier'].isin(TIERS)]
    .groupby(['grid_id', 'Date', 'Severity_Tier'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure all 4 tier columns exist
for t in TIERS:
    if t not in df_daily.columns:
        df_daily[t] = 0

df_daily = df_daily[['grid_id', 'Date'] + TIERS].copy()
df_daily = df_daily.sort_values(['grid_id', 'Date']).reset_index(drop=True)

print(f'Daily panel shape: {df_daily.shape}')
print('\nClass balance (positive rate = at least 1 crime that day):')
for i, t in enumerate(TIERS):
    rate = (df_daily[t] > 0).mean()
    label = ['Severe imbalance → SMOTE needed',
             'Balanced → no action needed',
             'Majority class → no action needed',
             'Mild imbalance → class_weight sufficient'][i]
    print(f'  Tier {i+1}: {rate:.1%}  ({label})')

## Step 3: Normalized Lag Features
**Why rolling MEAN instead of SUM?**
- Sum: `tier2_lag_7d = 42` (Chicago dense grid) vs `= 3` (sparse rural area)
- Mean: `tier2_lag_7d_mean = 6.0/day` vs `= 0.4/day` — still scale-different but interpretable
- Combined with surge ratio → fully scale-independent

Windows: **7d** (short-term), **30d** (medium-term baseline), **90d** (long-range trend)

`.shift(1)` ensures today's crime is excluded → **no data leakage**


In [ ]:
print('Creating normalized lag features (7d, 30d, 90d mean)...')

for i, tier in enumerate(TIERS):
    n = i + 1
    grp = df_daily.groupby('grid_id')[tier]

    df_daily[f'tier{n}_lag_7d_mean'] = grp.transform(
        lambda x: x.shift(1).rolling(7,  min_periods=1).mean()
    )
    df_daily[f'tier{n}_lag_30d_mean'] = grp.transform(
        lambda x: x.shift(1).rolling(30, min_periods=1).mean()
    )
    df_daily[f'tier{n}_lag_90d_mean'] = grp.transform(
        lambda x: x.shift(1).rolling(90, min_periods=1).mean()
    )
    print(f'  Tier {n} done.')

df_daily = df_daily.fillna(0)
print('Lag features complete.')

## Step 4: Surge Ratio, Trend & Cross-Tier Features
| Feature | Formula | Meaning |
|---|---|---|
| `surge_ratio` | 7d_mean / 30d_mean | Is crime spiking vs recent norm? |
| `trend` | 7d_mean − 30d_mean | Is crime rising or falling? |
| `violence_property_ratio` | tier2 / tier3 | Neighbourhood shifting to violent? |
| `disorder_to_violence` | tier4 / tier2 | Public disorder predicting violence? |

**Why surge ratio generalizes:** `2.0` means "double the usual rate" — same interpretation
whether the area averages 1 crime/day or 20 crimes/day.


In [ ]:
# ── Surge Ratio ──────────────────────────────────────────────────────────
# 7d_mean / 30d_mean → scale-independent spike detector
# clip lower=0.01 avoids divide-by-zero; clip upper=10 removes extreme outliers
for i in range(1, 5):
    df_daily[f'tier{i}_surge_ratio'] = (
        df_daily[f'tier{i}_lag_7d_mean'] /
        df_daily[f'tier{i}_lag_30d_mean'].clip(lower=0.01)
    ).clip(upper=10.0)

# ── Trend ─────────────────────────────────────────────────────────────────
# Positive = crime trending up, Negative = trending down
for i in range(1, 5):
    df_daily[f'tier{i}_trend'] = (
        df_daily[f'tier{i}_lag_7d_mean'] - df_daily[f'tier{i}_lag_30d_mean']
    )

# ── Cross-Tier Ratios ─────────────────────────────────────────────────────
# Violence vs Property balance
df_daily['violence_property_ratio'] = (
    df_daily['tier2_lag_7d_mean'] /
    df_daily['tier3_lag_7d_mean'].clip(lower=0.01)
).clip(upper=10.0)

# Public disorder → violence escalation signal
df_daily['disorder_to_violence'] = (
    df_daily['tier4_lag_7d_mean'] /
    df_daily['tier2_lag_7d_mean'].clip(lower=0.01)
).clip(upper=10.0)

print('Surge ratios, trends, cross-tier features created.')

## Step 5: Cyclical Time Features

In [ ]:
df_daily['month_sin'] = np.sin(2 * np.pi * df_daily['Date'].dt.month      / 12)
df_daily['month_cos'] = np.cos(2 * np.pi * df_daily['Date'].dt.month      / 12)
df_daily['dow_sin']   = np.sin(2 * np.pi * df_daily['Date'].dt.dayofweek  / 7)
df_daily['dow_cos']   = np.cos(2 * np.pi * df_daily['Date'].dt.dayofweek  / 7)
print('Cyclical features created.')

## Step 6: Binary Targets

In [ ]:
df_daily['y_tier1'] = (df_daily['Tier 1 - Lethal']           > 0).astype(int)
df_daily['y_tier2'] = (df_daily['Tier 2 - Personal Violence'] > 0).astype(int)
df_daily['y_tier3'] = (df_daily['Tier 3 - Property']          > 0).astype(int)
df_daily['y_tier4'] = (df_daily['Tier 4 - Public Order']      > 0).astype(int)
print('Binary targets created.')
for col in TARGET_COLS:
    print(f'  {col}: {df_daily[col].mean():.1%} positive rate')

## Step 7: Composite Risk — Correlation Weights
Weights are computed on **training data only (2015–2024)** to prevent leakage.
Then applied to the full dataset (including 2025).

Each target gets its own set of weights → model gets a pre-combined risk signal
tuned to what it is trying to predict.


In [ ]:
# Use training data only to compute weights
train_mask    = df_daily['Date'].dt.year <= 2024
df_train_only = df_daily[train_mask].copy()
lag_7d_cols   = [f'tier{i}_lag_7d_mean' for i in range(1, 5)]

composite_weights = {}
print('Correlation-based composite weights (computed on 2015-2024 only):')
print('-' * 65)

for target in TARGET_COLS:
    corr    = df_train_only[lag_7d_cols + [target]].corr()[target][lag_7d_cols].abs()
    total   = corr.sum()
    weights = (corr / total).values if total > 0 else np.array([0.25]*4)
    composite_weights[target] = weights
    print(f'  {target}:')
    for col, w in zip(lag_7d_cols, weights):
        print(f'    {col:30s}: {w:.3f}')

# Apply to full dataset (2015-2025)
for target in TARGET_COLS:
    n = target[-1]
    w = composite_weights[target]
    df_daily[f'composite_risk_tier{n}'] = (
        df_daily['tier1_lag_7d_mean'] * w[0] +
        df_daily['tier2_lag_7d_mean'] * w[1] +
        df_daily['tier3_lag_7d_mean'] * w[2] +
        df_daily['tier4_lag_7d_mean'] * w[3]
    )

# Save weights for NIBRS testing notebook
joblib.dump(composite_weights, os.path.join(SAVE_DIR, 'composite_weights.joblib'))
print('\nWeights saved.')

## Step 8: Define Final Feature List (30 features)

In [ ]:
FEATURES = (
    # Group 1: Cyclical time features (4)
    ['month_sin', 'month_cos', 'dow_sin', 'dow_cos'] +
    # Group 2: Normalized 7d rolling mean (4)
    [f'tier{i}_lag_7d_mean'  for i in range(1, 5)] +
    # Group 3: Normalized 30d rolling mean (4)
    [f'tier{i}_lag_30d_mean' for i in range(1, 5)] +
    # Group 4: Long-range 90d rolling mean (4)
    [f'tier{i}_lag_90d_mean' for i in range(1, 5)] +
    # Group 5: Surge ratio — 7d vs 30d (4)
    [f'tier{i}_surge_ratio'  for i in range(1, 5)] +
    # Group 6: Trend — 7d minus 30d (4)
    [f'tier{i}_trend'        for i in range(1, 5)] +
    # Group 7: Cross-tier ratios (2)
    ['violence_property_ratio', 'disorder_to_violence'] +
    # Group 8: Composite risk per target (4)
    [f'composite_risk_tier{i}' for i in range(1, 5)]
)

print(f'Total features: {len(FEATURES)}')
groups = [
    ('Cyclical time',         4),
    ('7d rolling mean',       4),
    ('30d rolling mean',      4),
    ('90d rolling mean',      4),
    ('Surge ratio',           4),
    ('Trend (7d-30d)',         4),
    ('Cross-tier ratios',     2),
    ('Composite risk',        4),
]
print(f'{"Group":<25} {"Count"}')
print('-' * 35)
for name, cnt in groups:
    print(f'  {name:<23} {cnt}')
print('-' * 35)
print(f'  {"TOTAL":<23} {sum(c for _,c in groups)}')

## Step 9: Save Processed Dataset

In [ ]:
KEEP_COLS = ['grid_id', 'Date'] + FEATURES + TARGET_COLS
df_out    = df_daily[KEEP_COLS].copy()

# Tag train / validation
df_out['split'] = np.where(df_out['Date'].dt.year <= 2024, 'train', 'validation')

# Null check
null_counts = df_out[FEATURES].isnull().sum()
if null_counts.sum() == 0:
    print('Null check: PASSED (0 nulls in all features)')
else:
    print('WARNING — nulls found:')
    print(null_counts[null_counts > 0])

# Save parquet
df_out.to_parquet(OUT_FILE, index=False)
print(f'\nSaved: {OUT_FILE}')
print(f'File size: {os.path.getsize(OUT_FILE)/1024/1024:.1f} MB')

# Save feature list
joblib.dump(FEATURES, os.path.join(SAVE_DIR, 'feature_names_v2.joblib'))
print(f'Feature list saved to Saved_Models/feature_names_v2.joblib')

## Step 10: Summary Report

In [ ]:
train_df = df_out[df_out['split'] == 'train']
val_df   = df_out[df_out['split'] == 'validation']

print('=' * 65)
print('  FEATURE ENGINEERING PIPELINE — COMPLETE SUMMARY')
print('=' * 65)
print(f'  Total records     : {len(df_out):>10,}')
print(f'  Train (2015-2024) : {len(train_df):>10,}')
print(f'  Val   (2025)      : {len(val_df):>10,}')
print(f'  Total features    : {len(FEATURES):>10}')
print()
print('  Class Balance:')
print(f'  {"Target":<12} {"Train positive":>16} {"Val positive":>14}')
print('  ' + '-' * 44)
for col in TARGET_COLS:
    tr = train_df[col].mean()
    vl = val_df[col].mean()
    print(f'  {col:<12} {tr:>15.1%} {vl:>13.1%}')
print()
print('  Feature value ranges (training set):')
print(train_df[FEATURES].describe().loc[['mean','std','min','max']].round(3).T.to_string())
print()
print('Pipeline complete. Load chicago_features_processed.parquet in model training notebook.')